# Setup
#### (Import Packages and Read Dataset)

In [13]:
# imports
import requests
import datetime
import pandas as pd
from py4j.java_gateway import java_import
from pyspark.sql.functions import to_timestamp, min, max, col, date_trunc, avg, last
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
base_path = r"abfss://datalake@bessstorage.dfs.core.windows.net/"
silver_path = base_path + r"silver/"
gold_path = base_path + r"gold/"

In [3]:
# determine delta table paths in Bronze directory
java_import(spark._jvm, 'org.apache.hadoop.fs.Path')

fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
status = fs.listStatus(spark._jvm.Path(silver_path))
delta_table_paths = [str(file.getPath()) for file in status if file.isDirectory()]
delta_table_paths

In [45]:
# read merged silver dataset as df
silver_path = "abfss://datalake@bessstorage.dfs.core.windows.net/silver/merged_delta"
silver_df = spark.read.format("delta").load(silver_path).toPandas()
silver_df.head(3)

# Group Cycles

In [46]:
silver_df = silver_df.sort_values('datetime_minute').reset_index(drop=True)

# Classify power state
def classify(power):
    if power <= -2000:
        return 'charging'
    elif power >= 2000:
        return 'discharging'
    else:
        return 'idle'

silver_df['state'] = silver_df['RTAC_P'].apply(classify)
silver_df.head(3)

In [47]:
cycles = []
in_cycle = False
cycle_type = None
start_idx = None

for i in range(len(silver_df)):
    current_state = silver_df.at[i, 'state']

    if not in_cycle:
        if current_state == 'charging':
            in_cycle = True
            cycle_type = 'charging'
            start_idx = i
        elif current_state == 'discharging':
            in_cycle = True
            cycle_type = 'discharging'
            start_idx = i
    else:
        # in a cycle
        if (cycle_type == 'charging' and current_state != 'charging') or \
            (cycle_type == 'discharging' and current_state != 'discharging'):
            # End of cycle
            end_idx = i - 1 if i > start_idx else i
            cycle_df = silver_df.iloc[start_idx:end_idx+1]

            if not cycle_df.empty:
                cycles.append({
                    'start_datetime_minute': cycle_df['datetime_minute'].iloc[0],
                    'end_datetime_minute': cycle_df['datetime_minute'].iloc[-1],
                    'cycle_type': cycle_type,
                    'cycle_duration': (cycle_df['datetime_minute'].iloc[-1] - cycle_df['datetime_minute'].iloc[0]).total_seconds() / 60,
                    'average_power': cycle_df['RTAC_P'].mean(),
                    'soc_change': cycle_df['SOC'].iloc[-1] - cycle_df['SOC'].iloc[0]
                })

            in_cycle = False
            cycle_type = None
            start_idx = None

# Catch if at end of data
if in_cycle and start_idx is not None:
    cycle_df = silver_df.iloc[start_idx:]
    if not cycle_df.empty:
        cycles.append({
            'start_datetime_minute': cycle_df['datetime_minute'].iloc[0],
            'end_datetime_minute': cycle_df['datetime_minute'].iloc[-1],
            'cycle_type': cycle_type,
            'cycle_duration': (cycle_df['datetime_minute'].iloc[-1] - cycle_df['datetime_minute'].iloc[0]).total_seconds() / 60,
            'average_power': cycle_df['power'].mean(),
            'soc_change': cycle_df['SOC'].iloc[-1] - cycle_df['SOC'].iloc[0]
        })

cycles = pd.DataFrame(cycles)
cycles.head(3)

# Write into Delta Table in Gold Directory

In [48]:
cycles = spark.createDataFrame(cycles)
cycles.write.format("delta").mode("overwrite").save(gold_path + "cycles_delta")